In [1]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/machine-learning-comp-432-project/sample_submission.csv
/kaggle/input/machine-learning-comp-432-project/train.csv
/kaggle/input/machine-learning-comp-432-project/test.csv


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [3]:
# Paths
train_path = "/kaggle/input/machine-learning-comp-432-project/train.csv"
test_path  = "/kaggle/input/machine-learning-comp-432-project/test.csv"

# Load CSVs
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

# Feature columns
feature_cols = [c for c in train_df.columns if c.startswith("feature")]

X = train_df[feature_cols].values
y = train_df["label"].values
X_test = test_df[feature_cols].values

NUM_FEATURES = X.shape[1]
NUM_CLASSES = 50  # no. of unique labels

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

Train: (115406, 502)
Test : (49460, 501)


In [4]:
class FeatureDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

# Split 90% train / 10% validation
train_size = int(0.9 * len(train_df))
val_size = len(train_df) - train_size

full_dataset = FeatureDataset(X, y)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)

test_dataset = FeatureDataset(X_test)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)


In [5]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

model = MLP(NUM_FEATURES, NUM_CLASSES)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
EPOCHS = 20
best_val_acc = 0

for epoch in range(EPOCHS):
    # ---- Train ----
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    # ---- Validation ----
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            outputs = model(xb)
            pred = outputs.argmax(dim=1).cpu().numpy()
            preds.extend(pred)
            targets.extend(yb.numpy())

    val_acc = accuracy_score(targets, preds)
    print(f"Epoch {epoch+1}/{EPOCHS} - Val Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

print("Best Validation Accuracy:", best_val_acc)

Epoch 1/20 - Val Accuracy: 0.4145
Epoch 2/20 - Val Accuracy: 0.4935
Epoch 3/20 - Val Accuracy: 0.5253
Epoch 4/20 - Val Accuracy: 0.5461
Epoch 5/20 - Val Accuracy: 0.5583
Epoch 6/20 - Val Accuracy: 0.5701
Epoch 7/20 - Val Accuracy: 0.5695
Epoch 8/20 - Val Accuracy: 0.5786
Epoch 9/20 - Val Accuracy: 0.5784
Epoch 10/20 - Val Accuracy: 0.5758
Epoch 11/20 - Val Accuracy: 0.5767
Epoch 12/20 - Val Accuracy: 0.5737
Epoch 13/20 - Val Accuracy: 0.5746
Epoch 14/20 - Val Accuracy: 0.5795
Epoch 15/20 - Val Accuracy: 0.5792
Epoch 16/20 - Val Accuracy: 0.5778
Epoch 17/20 - Val Accuracy: 0.5802
Epoch 18/20 - Val Accuracy: 0.5769
Epoch 19/20 - Val Accuracy: 0.5766
Epoch 20/20 - Val Accuracy: 0.5805
Best Validation Accuracy: 0.5804523004938913


In [7]:
# Load best model weights
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# Load test.csv
test_df = pd.read_csv("/kaggle/input/machine-learning-comp-432-project/test.csv")
test_ids = test_df["id"]

# Scale features
X_test = test_df.drop(columns=["id"]).values
X_test = scaler.transform(X_test)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

# Predict
with torch.no_grad():
    outputs = model(X_test_tensor)
    predictions = torch.argmax(outputs, dim=1).cpu().numpy()

# Create submission dataframe
submission = pd.DataFrame({
    "id": test_ids,
    "label": predictions
})

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("Submission file saved to:", submission_path)
submission.head()


Submission file saved to: /kaggle/working/submission.csv


,id,label
0,0,0
1,1,44
2,2,3
3,3,11
4,4,0
